In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
# we need this imports here
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms

def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
from PIL import Image
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import os
image_paths = []
mask_paths = []

# Loop through all folders, the data looks messy so i need to make sperate paths somthing like this
for underwater_folder in os.listdir(dataset_root):
  patient_path = os.path.join(dataset_root, patient_folder)

  if os.path.isdir(patient_path):
    for foldername in os.listdir(patient_path):
      # TODO: Check if the file is a mask (contains "_mask")
      # If it's a mask, add to mask_paths
      # Otherwise, if it's a .tif file, add to image_paths

      # YOUR CODE HERE
      if "mask" in foldername:
        mask_paths.append(os.path.join(patient_path, filename))
      elif foldername.endswith(".png"):
        image_paths.append(os.path.join(patient_path, filename))

print(f"Total images: {len(image_paths)}")
print(f"Total masks: {len(mask_paths)}")


class UnderwaterImagery(Dataset):
    def __init__(self, root_dir, csv_file, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.metadata = pd.read_csv(csv_file)
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
      #Return the number of samples in the dataset
        return len(self.metadata)

    def __getitem__(self, idx):
      #Load the image and mask at index idx
    # Use Image.open() and convert image to "RGB", mask to "L" (grayscale)

        img_path = os.path.join(self.root_dir, "Image", self.metadata.iloc[idx, 0])   # Use the csv to get paths
        mask_path = os.path.join(self.root_dir, "Mask", self.metadata.iloc[idx, 1])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Convert mask to grayscale (1 channel = binary segmentation)
# Apply transforms
        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        # Replace mask values with remapped values
        mask = remap_mask(mask) # Convert to binary mask

        return image, mask       # In image classification datasets, we return image and label. Here, we return image and mask

# Image transforms (Resize, ToTensor, Normalize with ImageNet stats)
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Standard ImageNet normalization
])

# Mask transforms (Resize, PILToTensor)
mask_transforms = transforms.Compose([
  # Add transforms
  Resize to (256, 256) with NEAREST interpolation, PILToTensor
  transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
  transforms.PILToTensor(),
])

# Split into train and test sets (80% train, 20% test)

train_images, test_images, train_masks, test_masks = train_test_split(
  image_paths, mask_paths, test_size=0.2, random_state=42
)

# Create Dataset objects
train_dataset = UnderwaterImagery(train_images, train_masks, transform=image_transforms, target_transform=mask_transforms)
test_dataset = UnderwaterImagery(test_images, test_masks, transform=image_transforms, target_transform=mask_transforms)

# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

#LOADING DATA
## **🔹 Splitting the Dataset into Train & Test**

# Define dataset paths
csv_path = os.path.join(path, "metadata.csv")

# Read metadata CSV
metadata = pd.read_csv(csv_path)

# Split dataset into 80% train, 20% test
train_data, test_data = train_test_split(metadata, test_size=0.2, random_state=42, shuffle=True)

# Save split CSVs in /content
# i know the data doesnt look like this but like 2 folders of images and masks i really dont know how to split them im so sick and cant think right agh
# i just know that i need to zip the images with thier masks and make a tuple in directory that bridge the image to its mask and then split that to test and train
train_path = os.path.join('/content', 'train.csv')
test_path = os.path.join('/content', 'test.csv')

train_data.to_csv(train_path, index=False)
test_data.to_csv(test_path, index=False)


# Load training dataset
train_dataset = UnderwaterImagery(root_dir=path, csv_file=train_path,
                                         transform=image_transforms, target_transform=mask_transforms)

# Load testing dataset
test_dataset = UnderwaterImagery(root_dir=path, csv_file=test_path,
                                        transform=image_transforms, target_transform=mask_transforms)

#create DataLoader for train, validation, and test sets
# Create Train & Test DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

# Check dataset sizes
print(f"Training Samples: {len(train_dataset)}, Testing Samples: {len(test_dataset)}")


#task3---------------------------------------------------------------------------------

import matplotlib.pyplot as plt
import numpy as np

def denormalize(img):
  mean = np.array([0.485, 0.456, 0.406])
  std = np.array([0.229, 0.224, 0.225])
  img = img.permute(1, 2, 0).numpy()  # CHW -> HWC
  img = img * std + mean
  img = np.clip(img, 0, 1)
  return img
## Just Looking at the data and trying too visualize it
# Function to denormalize images (We cannot show normalized images. We have to reverse normalizaion first.)
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Display some images with their masks
for i in range(3):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.permute(1,2,0), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()


In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# YOUR CODE HERE
model = smp.Unet(
  encoder_name="efficientnet-b1",
  encoder_weights="imagenet",
  in_channels=3,
  classes=1,
).to(device)

model = model.to(device)

In [ ]:
# TO DO
import torch.optim as optim
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
  model.train()
  total_loss = 0

  for images, masks in tqdm(dataloader):
    # Move data to device
    images, masks = images.to(device), masks.to(device).float()

    # TODO: Complete the training step
    # 1. Forward pass
    # 2. Compute loss
    # 3. Zero gradients
    # 4. Backward pass
    # 5. Update weights

    # YOUR CODE HERE
    outputs = model(images)
    loss = criterion(outputs, masks)
    optimizer.zero_grad()
    loss.backward() #computes the gradients of the loss with respect to model parameters using backpropagation.
    optimizer.step() #updates the model parameters using the computed gradients to minimize the loss

    total_loss += loss.item()

  return total_loss / len(dataloader)


#validation function

def validate(model, dataloader, criterion, device):
  model.eval()#disables Dropout and uses fixed statistics in Batch Normalization to ensure consistent evaluation results
  total_loss = 0

  with torch.no_grad():
    for images, masks in dataloader:
      images, masks = images.to(device), masks.to(device).float()

      # TODO: Complete the validation step
      # 1. Forward pass
      # 2. Compute loss

      # YOUR CODE HERE
      outputs = model(images)
      loss = criterion(outputs, masks)

      total_loss += loss.item()

  return total_loss / len(dataloader)

from torch import nn

# YOUR CODE HERE
#CrossEntropyLoss combines Softmax and Negative Log Likelihood, making it suitable for multi-class classification problems
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

num_epochs = 10  # Train for 5 epochs

In [ ]:
# TO DO

# Run training
train_losses = []
val_losses = []

for epoch in range(num_epochs):
  train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
  val_loss = validate(model, test_loader, criterion, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# TO DO

# TODO: Task 8 - Visualize predictions on test images
import random

model.eval()

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

# Get random test samples
indices = random.sample(range(len(test_dataset)), 4)

for i, idx in enumerate(indices):
  image, mask = test_dataset[idx]

  # TODO: Get model prediction
  # 1. Add batch dimension: image.unsqueeze(0)
  # 2. Move to device
  # 3. Get prediction: model(image)
  # 4. Apply sigmoid to get probabilities
  # 5. Threshold at 0.5 to get binary mask

  with torch.no_grad():
    input_tensor = image.unsqueeze(0).to(device)
    output = model(input_tensor)
    pred = torch.sigmoid(output)
    pred = (pred > 0.5).float().cpu()

  # Display results
  axes[i, 0].imshow(denormalize(image))
  axes[i, 0].set_title("")
  axes[i, 0].axis("off")

  axes[i, 1].imshow(mask.squeeze(), cmap="gray")
  axes[i, 1].set_title("Ground Truth")
  axes[i, 1].axis("off")

  axes[i, 2].imshow(pred.squeeze(), cmap="gray")
  axes[i, 2].set_title("Prediction")
  axes[i, 2].axis("off")

plt.tight_layout()
plt.show()